# Bem-vindo à Análise Preditiva de Demanda e Ocupação na Aviação Comercial

## Contexto e Apresentação da Base de Dados
A rentabilidade e a eficiência da aviação comercial dependem drasticamente da otimização de espaço e da previsão de demanda. Este projeto acadêmico tem como objetivo explorar dados históricos e construir um modelo preditivo (Regressão) para antecipar o volume de passageiros em voos no Brasil. 

Para isso, utilizaremos a base de **Dados Estatísticos do Transporte Aéreo**, fornecida pela Agência Nacional de Aviação Civil (ANAC). Diferente das bases focadas em pontualidade, esta base de dados tem seu foco na ocupação e no peso comercial dos voos, trazendo um raio-x completo do fluxo de passageiros no país.

### Estrutura das Variáveis Principais (Dicionário de Dados)
Para a construção do nosso modelo preditivo, filtraremos a base original focando nas variáveis que mais influenciam a demanda:
* **sg_empresa_iata:** A companhia aérea operadora (ex: LA, G3, AD).
* **nm_mes_referencia e nm_dia_semana_referencia:** Indicadores temporais essenciais para capturar a sazonalidade (alta e baixa temporada).
* **sg_iata_origem e sg_iata_destino:** Aeroportos que definem a rota operada.
* **nr_seats_offered:** A oferta estrutural do avião (quantos assentos a aeronave possui).
* **nr_passag_pagos:** A quantidade real de passageiros pagantes que embarcaram (a nossa Variável Alvo).

---

## A Problemática: O Custo do Assento Vazio e a Gestão de Malha
Na indústria aérea, o "produto" vendido (o assento no voo) é altamente perecível. No exato instante em que a porta da aeronave se fecha e o avião é tratorado (pushback), qualquer assento vazio representa uma receita perdida que nunca mais poderá ser recuperada. Em contrapartida, o custo operacional daquele voo — que inclui querosene de aviação, leasing da aeronave, tripulação e taxas aeroportuárias — permanece alto e fixo, independentemente de o avião estar com 50% ou 100% de ocupação.

**O Objetivo do Modelo:**
Diante dessa dinâmica agressiva de custos, o desafio deste projeto é aplicar técnicas de Aprendizado de Máquina (Machine Learning) para descobrir padrões históricos de lotação e treinar um algoritmo capaz de prever a ocupação futura de uma rota. 

Antecipar a demanda de passageiros é o coração do *Revenue Management* (Gestão de Receitas) na aviação: permite que as empresas apliquem precificação dinâmica assertiva, aloquem aeronaves maiores para rotas que vão bombar e evitem o desperdício de voar com "ar" no porão e na cabine.

# Análise Exploratória de Dados
Vamos começar fatiando nosso arquivo csv para trabalharmos apenas com as colunas que nos interessam.

In [17]:
import pandas as pd

path = "./base/anac_brazil.csv"
exit = 'dados_aviacao.csv'

target = [
    'sg_empresa_iata',
    'nm_mes_referencia',
    'nm_dia_semana_referencia',
    'sg_iata_origem',
    'sg_iata_destino',
    'nr_assentos_ofertados',
    'nr_passag_pagos'
]

chunk_size = 100000
first_chunk = True

print("Iniciando...")

try:
    for chunk in pd.read_csv(path, sep=',', encoding='utf-8', chunksize=chunk_size, usecols=target):
        chunk = chunk.dropna(subset=['nr_passag_pagos', 'nr_assentos_ofertados'])

        if first_chunk:
            chunk.to_csv(exit, index=False, encoding='utf-8')
            first_chunk = False
        else:
            chunk.to_csv(exit, index=False, mode='a', header=False, encoding='utf-8')

    print("Concluído com sucesso!")

except ValueError as e:
    print(f"Erro ao ler as colunas. Detalhes: {e}")


Iniciando...


/tmp/ipykernel_183960/2217160333.py:22: DtypeWarning: Columns (0: sg_iata_origem) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, sep=',', encoding='utf-8', chunksize=chunk_size, usecols=target):


Concluído com sucesso!


Agora, com o dataset resumido em mãos, vamos verificar as informações, gerar um resumo e conferir os dados ausentes:

In [21]:
df = pd.read_csv('dados_aviacao.csv', sep=',')

print("--- Informações do Dataset ---")
df.info()
print('\n')

print("--- Estatísticas Descritivas ---")
display(df.describe())
print('\n')

print("--- Valores Ausentes por Coluna ---")
display(df.isnull().sum())

--- Informações do Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 22119967 entries, 0 to 22119966
Data columns (total 7 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   sg_empresa_iata           str    
 1   nm_mes_referencia         str    
 2   nm_dia_semana_referencia  str    
 3   sg_iata_origem            str    
 4   sg_iata_destino           str    
 5   nr_assentos_ofertados     float64
 6   nr_passag_pagos           float64
dtypes: float64(2), str(5)
memory usage: 1.2 GB


--- Estatísticas Descritivas ---


,nr_assentos_ofertados,nr_passag_pagos
count,2.211997e+07,2.211997e+07
mean,1.407486e+02,1.015275e+02
std,6.373404e+01,6.044836e+01
min,0.000000e+00,0.000000e+00
25%,1.100000e+02,5.700000e+01
50%,1.440000e+02,1.020000e+02
75%,1.760000e+02,1.410000e+02
max,9.950000e+02,1.392000e+03




--- Valores Ausentes por Coluna ---


sg_empresa_iata             559699
nm_mes_referencia                0
nm_dia_semana_referencia         0
sg_iata_origem              319067
sg_iata_destino             320900
nr_assentos_ofertados            0
nr_passag_pagos                  0
dtype: int64